# Lab: Generative AI for Data Science
## Student: Mohamed Abdirahman
## Course: DSF-FTR16 | Module 3

This lab explores three real-world scenarios using AI-generated code.
Each scenario includes:
- The prompt used
- The AI-generated code
- Follow-up prompts
- Final working solution

## Scenario 1: Retail Inventory Analysis

### My Prompt:
"Act as a senior data scientist. I have a retail inventory DataFrame with columns: 
product_id, category, stock_level, last_restock_date, sales_last_30_days, 
supplier_lead_time, unit_cost. Generate code to calculate inventory turnover rates, 
identify slow-moving items, predict potential stockouts, and create visualizations."

### Why I wrote the prompt this way:
I assigned the AI a role (senior data scientist) to get more professional output.
I listed all column names clearly so the AI understood the exact data structure.
I specified all four tasks clearly to get complete coverage.

In [3]:
# Scenario 1: Retail Inventory Analysis
# Generated Code

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Sample Data
np.random.seed(42)
n = 100

data = {
    'product_id': [f'P{i:03d}' for i in range(n)],
    'category': np.random.choice(['Electronics', 'Clothing', 'Food', 'Home'], n),
    'stock_level': np.random.randint(10, 500, n),
    'last_restock_date': [datetime.now() - timedelta(days=np.random.randint(1, 60)) 
                          for _ in range(n)],
    'sales_last_30_days': np.random.randint(0, 200, n),
    'supplier_lead_time': np.random.randint(3, 21, n),
    'unit_cost': np.random.uniform(5, 500, n).round(2)
}

df = pd.DataFrame(data)

# 1. Inventory Turnover Rate
df['daily_sales_rate'] = df['sales_last_30_days'] / 30
df['turnover_rate'] = np.where(
    df['stock_level'] > 0,
    df['sales_last_30_days'] / df['stock_level'],
    0
)

# 2. Slow-moving items (bottom 25%)
threshold = df['turnover_rate'].quantile(0.25)
df['slow_moving'] = df['turnover_rate'] <= threshold

# 3. Predict Stockouts
df['days_until_stockout'] = np

### Follow-up Prompts:
1. "The visualization is not showing in the notebook, how do I fix it?"
2. "Can you add data validation to check for negative stock levels?"

### What I learned:
- Setting matplotlib backend to 'Agg' fixes display issues
- Using np.where() handles division by zero when stock_level is 0
- NTILE/quantile(0.25) effectively identifies slow-moving items

### Final Results:
- Inventory turnover rate calculated for all 100 products
- Slow-moving items identified using bottom 25% threshold
- Stockout risk flagged where days_until_stockout <= supplier_lead_time
- Two visualizations saved successfully

## Scenario 2: Website Analytics Debug

### My Prompt:
"I have a buggy Python function that calculates user engagement metrics.
It produces impossible results: bounce rates over 100% and negative session times.

Here is the broken code:

def analyze_user_engagement(logs_df):
    metrics = {
        'bounce_rate': logs_df.groupby('session_id')['page_views'].apply(
            lambda x: x == 1).mean(),
        'avg_session_time': logs_df.groupby('session_id')['duration'].sum(),
        'pages_per_session': logs_df.groupby('session_id')['page_views'].mean()
    }
    return metrics

Please identify the bugs, explain why they cause wrong results, fix them,
add data validation, and create summary visualizations."

### Why I wrote the prompt this way:
I included the actual broken code so the AI could see exactly what was wrong.
I described the symptoms (bounce rate over 100%) to help the AI identify the bugs.
I asked for explanations so I could understand the fixes, not just copy them.

In [4]:
# Scenario 2: Website Analytics Debug
# Generated Code

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Sample Data
np.random.seed(42)
n = 500
logs_df = pd.DataFrame({
    'session_id': np.random.randint(1, 101, n),
    'page_views':  np.random.randint(1, 10, n),
    'duration':    np.random.randint(10, 600, n),
    'device_type': np.random.choice(['mobile','desktop','tablet'], n)
})

def analyze_user_engagement(logs_df):
    """
    BUGS FIXED:
    - bounce_rate: original code compared each row value to 1 and averaged
      booleans giving values over 100%. Fix: find sessions where total 
      page_views == 1 and calculate percentage.
    - avg_session_time: original used .sum() per session instead of .mean()
      across all sessions, inflating the result.
    - No validation: negative durations were silently accepted.
    """
    # Data Validation
    if logs_df.isnull().any().any():
        print("Warning: Null values found - dropping affected rows")
        logs_df = logs_df.dropna()

    if (logs_df['duration'] < 0).any():
        print("Warning: Negative durations found - removing invalid rows")
        logs_df = logs_df[logs_df['duration'] >= 0]

    if (logs_df['page_views'] < 1).any():
        print("Warning: page_views less than 1 found - removing invalid rows")
        logs_df = logs_df[logs_df['page_views'] >= 1]

    # Fixed Calculations
    session_data = logs_df.groupby('session_id').agg(
        total_pages=('page_views', 'sum'),
        total_duration=('duration', 'sum')
    )

    # FIX 1: bounce_rate = % of sessions with only 1 page view
    bounce_rate = (session_data['total_pages'] == 1).mean() * 100

    # FIX 2: avg_session_time = mean duration across all sessions
    avg_session_time = session_data['total_duration'].mean()

    # pages_per_session = mean pages across all sessions
    pages_per_session = session_data['total_pages'].mean()

    metrics = {
        'bounce_rate': round(bounce_rate, 2),
        'avg_session_time_seconds': round(avg_session_time, 2),
        'pages_per_session': round(pages_per_session, 2)
    }

    device_metrics = logs_df.groupby('device_type').agg(
        session_count=('session_id', 'nunique'),
        avg_duration=('duration', 'mean'),
        total_page_views=('page_views', 'sum')
    ).round(2)

    return metrics, device_metrics

metrics, device_metrics = analyze_user_engagement(logs_df)

print("── Engagement Metrics ──────────────────")
for k, v in metrics.items():
    print(f"  {k}: {v}")
print("\n── Device Metrics ──────────────────────")
print(device_metrics)

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

device_metrics['avg_duration'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Avg Session Duration by Device')
axes[0].set_ylabel('Seconds')

device_metrics['total_page_views'].plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title('Total Page Views by Device')
axes[1].set_ylabel('Page Views')

plt.tight_layout()
plt.savefig('engagement_analysis.png')
plt.show()

── Engagement Metrics ──────────────────
  bounce_rate: 0.0
  avg_session_time_seconds: 1462.27
  pages_per_session: 24.48

── Device Metrics ──────────────────────
             session_count  avg_duration  total_page_views
device_type                                               
desktop                 79        291.33               834
mobile                  75        281.64               877
tablet                  79        296.03               713


### Follow-up Prompts:
1. "Can you explain exactly why the original bounce_rate calculation 
    was giving values over 100%?"
2. "Can you add a validation step to catch negative session durations?"

### Bugs Identified and Fixed:
| Bug | Original Code | Problem | Fix |
|-----|--------------|---------|-----|
| bounce_rate | .apply(lambda x: x==1).mean() | Compared each row to 1, averaged booleans | Count sessions with total pages == 1 |
| avg_session_time | .sum() per session | Summed instead of averaged | Use .mean() after grouping |
| No validation | None | Silent errors with bad data | Added checks for nulls and negatives |

### Final Results:
- Bounce rate now correctly shows percentage between 0-100%
- Average session time is now a meaningful average in seconds
- Data validation catches and removes invalid records
- Device metrics show proper unique session counts

## Scenario 3: Customer Segmentation Query

### My Prompt:
"I need an optimized SQL query for customer segmentation using three tables:
- user_activity (user_id, last_login_date, feature_usage_count, account_type)
- transactions (transaction_id, user_id, transaction_date, amount, platform)
- user_preferences (user_id, communication_preference, interface_theme, 
  notification_settings)

Write a SQL query using CTEs to:
1. Filter active users who logged in within the last 30 days
2. Identify high-value customers in the top 20% by total spending
3. Join user preference trends for these customers
4. Use NTILE for percentile calculations
5. Explain why you chose CTEs over subqueries"

### Why I wrote the prompt this way:
I included the full database schema with exact column names so the AI
could write accurate field references. I specified CTEs explicitly to
get an optimized query. I asked for an explanation of optimization
choices to deepen my understanding.

In [5]:
# Scenario 3: Customer Segmentation Query
# Since we don't have a real database, we simulate the SQL using pandas
# The actual SQL query is documented below in comments

import pandas as pd
import numpy as np

# ── Simulate the three database tables ───────────────────────────────────────
np.random.seed(42)
n_users = 200

user_activity = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'last_login_date': pd.date_range(end='2026-06-03', periods=n_users, freq='D')[::-1],
    'feature_usage_count': np.random.randint(0, 100, n_users),
    'account_type': np.random.choice(['free', 'premium', 'enterprise'], n_users)
})

transactions = pd.DataFrame({
    'transaction_id': range(1, 501),
    'user_id': np.random.randint(1, n_users + 1, 500),
    'transaction_date': pd.date_range(end='2026-06-03', periods=500, freq='D')[::-1],
    'amount': np.random.uniform(10, 1000, 500).round(2),
    'platform': np.random.choice(['web', 'mobile', 'desktop'], 500)
})

user_preferences = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'communication_preference': np.random.choice(['email', 'sms', 'push'], n_users),
    'interface_theme': np.random.choice(['light', 'dark'], n_users),
    'notification_settings': np.random.choice(['all', 'important', 'none'], n_users)
})

# ── SQL Query (documented) ────────────────────────────────────────────────────
sql_query = """
-- CTE 1: Active users (logged in last 30 days)
WITH active_users AS (
    SELECT
        user_id,
        last_login_date,
        feature_usage_count,
        account_type,
        DATEDIFF(DAY, last_login_date, GETDATE()) AS days_since_login
    FROM user_activity
    WHERE last_login_date >= DATEADD(DAY, -30, GETDATE())
),

-- CTE 2: Total spending per user
user_spending AS (
    SELECT
        user_id,
        SUM(amount) AS total_spent,
        COUNT(*) AS transaction_count
    FROM transactions
    GROUP BY user_id
),

-- CTE 3: Top 20% spenders using NTILE
spending_percentile AS (
    SELECT
        user_id,
        total_spent,
        transaction_count,
        NTILE(5) OVER (ORDER BY total_spent DESC) AS spending_quintile
    FROM user_spending
),

-- CTE 4: High-value customers (top 20%)
high_value_customers AS (
    SELECT user_id, total_spent, transaction_count
    FROM spending_percentile
    WHERE spending_quintile = 1
)

-- Final: Join all CTEs with preferences
SELECT
    au.user_id,
    au.account_type,
    au.feature_usage_count,
    au.days_since_login,
    hvc.total_spent,
    hvc.transaction_count,
    up.communication_preference,
    up.interface_theme,
    up.notification_settings
FROM active_users au
INNER JOIN high_value_customers hvc ON au.user_id = hvc.user_id
LEFT JOIN user_preferences up ON au.user_id = up.user_id
ORDER BY hvc.total_spent DESC;
"""

print("SQL Query for Customer Segmentation:")
print(sql_query)

# ── Pandas equivalent of the SQL query ───────────────────────────────────────
# Step 1: Active users (last 30 days)
cutoff_date = pd.Timestamp('2026-06-03') - pd.Timedelta(days=30)
active_users = user_activity[user_activity['last_login_date'] >= cutoff_date].copy()

# Step 2: Total spending per user
user_spending = transactions.groupby('user_id').agg(
    total_spent=('amount', 'sum'),
    transaction_count=('amount', 'count')
).reset_index()

# Step 3: Top 20% spenders
threshold = user_spending['total_spent'].quantile(0.80)
high_value = user_spending[user_spending['total_spent'] >= threshold]

# Step 4: Join all together
result = active_users.merge(high_value, on='user_id', how='inner')
result = result.merge(user_preferences, on='user_id', how='left')
result = result.sort_values('total_spent', ascending=False)

print("\nCustomer Segmentation Results:")
print(f"Active users: {len(active_users)}")
print(f"High-value customers: {len(high_value)}")
print(f"Active + High-value customers: {len(result)}")
print("\nTop 10 customers:")
print(result[['user_id','account_type','total_spent',
              'communication_preference','interface_theme']].head(10))

SQL Query for Customer Segmentation:

-- CTE 1: Active users (logged in last 30 days)
WITH active_users AS (
    SELECT
        user_id,
        last_login_date,
        feature_usage_count,
        account_type,
        DATEDIFF(DAY, last_login_date, GETDATE()) AS days_since_login
    FROM user_activity
    WHERE last_login_date >= DATEADD(DAY, -30, GETDATE())
),

-- CTE 2: Total spending per user
user_spending AS (
    SELECT
        user_id,
        SUM(amount) AS total_spent,
        COUNT(*) AS transaction_count
    FROM transactions
    GROUP BY user_id
),

-- CTE 3: Top 20% spenders using NTILE
spending_percentile AS (
    SELECT
        user_id,
        total_spent,
        transaction_count,
        NTILE(5) OVER (ORDER BY total_spent DESC) AS spending_quintile
    FROM user_spending
),

-- CTE 4: High-value customers (top 20%)
high_value_customers AS (
    SELECT user_id, total_spent, transaction_count
    FROM spending_percentile
    WHERE spending_quintile = 1
)

-- Final: 

### Follow-up Prompts:
1. "Why did you use CTEs instead of subqueries for this query?"
2. "Can you show me the pandas equivalent of the SQL query?"
3. "How does NTILE(5) work for finding the top 20% of spenders?"

### Query Optimization Choices Explained:
| Choice | Reason |
|--------|--------|
| CTEs | More readable than nested subqueries, easier to debug each step |
| NTILE(5) | Divides data into 5 equal groups, quintile 1 = top 20% |
| INNER JOIN | Ensures only users in both active AND high-value sets appear |
| LEFT JOIN on preferences | Keeps customers even if preferences are missing |
| Filter early in CTEs | Reduces rows before joins, improving performance |

### How percentile calculations were handled:
- NTILE(5) splits all users into 5 equal buckets by spending
- Bucket 1 = top 20% spenders (high-value customers)
- In pandas equivalent: quantile(0.80) achieves the same result

### Final Results:
- Active users filtered to last 30 days
- High-value customers identified as top 20% by total spending
- User preferences joined to show communication and theme trends
- Results sorted by total_spent descending

## Summary & Reflections

### What I learned from this lab:

| Scenario | Key Insight |
|----------|-------------|
| Retail Inventory | Assigning an AI a role (senior data scientist) produces more professional and complete code |
| Website Analytics | Including the broken code in the prompt helps AI identify bugs more accurately |
| Customer Segmentation | Specifying exact column names and table schemas produces more accurate SQL |

### Prompt Engineering Lessons:
1. **Be specific** — include column names, data types, and expected outputs
2. **Assign a role** — "Act as a senior data scientist" improves output quality
3. **Include context** — showing broken code helps AI debug more accurately
4. **Iterate** — follow-up prompts refine and improve the initial output
5. **Ask for explanations** — understanding the generated code is as important as the code itself

### Challenges Encountered:
- Matplotlib required backend setting (Agg) to work outside interactive mode
- SQL query needed pandas equivalent for testing without a real database
- Bounce rate bug was subtle — required careful reading of the original logic

### Conclusion:
Prompt engineering is a critical skill for data scientists. The quality 
of AI-generated code depends heavily on how well the problem is defined, 
contextualized, and constrained in the prompt.